# PyTorch Foundations 02 - The numbers survived. Did their meaning?

Part 01's sensor loop learned a curved correction. Its tables were simple:
examples down the rows, one number across. That made training visible, but
it left a dangerous habit intact: treating any shape-compatible operation
as the right operation.

**Guiding question:** Can you say which example, position, or feature every
result belongs to?

| When something goes wrong | Smallest check here |
|---|---|
| Selecting one example loses the batch axis | Compare an integer index with a slice |
| A reshape keeps all values but groups them wrongly | Split features, move axes, then undo both |
| An offset lands on the wrong role | Contrast feature and position broadcasting |
| A score is confused with its feedback | Audit a tiny derivative and accumulated gradients |
| A parameter table is mistaken for its inputs | Follow IDs through lookup, scores, and backward |

![Two four-word examples keep their example and position axes as IDs fetch six-feature rows and a projection produces ten candidate scores per position.](../images/02-word-addresses-and-axes.png)

We will run tensor operations and a gradient-wiring check. We will not train
a useful language model or implement attention.
[Handwritten theory](02-tensors-and-autograd-theory.md).


## Borrow one extra axis, not a whole language task

The sensor had examples and features. Two short strips, **"the cat sat on"**
and **"the dog sat on"**, add positions *inside* an example. That makes a lost
or moved axis much easier to see.

The local vocabulary below assigns `the=0`, `cat=1`, `sat=2`, `on=3`, `dog=4`.
So the input IDs are `[0, 1, 2, 3]` and `[0, 4, 2, 3]`: two rows, four
addresses each. ID 4 means "fetch row 4", not "four times as much word".

For the first shape experiments, `states = randn(2, 4, 6)` creates **separate
random features**; it does not look these IDs up yet. The real lookup comes
near the end. Neither set of random features knows English.

Setup fixes the random starting draw and prints the device. Device says where
numbers live; it doesn't change what the axes mean. Part 03 returns to the
sensor rows so parameter ownership can be the next new problem.


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")

# -- Imports, reproducibility, and device -------------------------------------
import torch

torch.manual_seed(11)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={device}")


## 1 - Count the roles before reaching for an API

One reading `5.5` has no axes. `[1, 2, 3]` has one. The two word strips need
two axes, and giving every position six features needs three:

```text
IDs      [2, 4]     = two examples, four positions each
states   [2, 4, 6]  = the same positions, six feature values each
```

There are eight IDs and 48 state values. Six features do not mean six
sentences, and the state tensor isn't a 48-parameter model.

Check the other labels too: IDs use integer `long` addresses; features use
floating-point values that can be combined smoothly. CPU/GPU placement must
agree for an operation. Matching shapes don't repair the wrong dtype or a
device mismatch.

The next cell prints rank, shape, dtype, and device together. Then we'll try
selecting a sentence without accidentally changing how later layers read it.


In [ ]:
# -- Inspect scalar, vector, table, and sequence contracts --------------------
scalar = torch.tensor(5.5)
vector = torch.tensor([1.0, 2.0, 3.0])
vocabulary = ["the", "cat", "sat", "on", "dog", "mat", "slept", "a", "bird", "flew"]
word_to_id = {word: index for index, word in enumerate(vocabulary)}
phrases = ["the cat sat on", "the dog sat on"]
token_ids = torch.tensor(
    [[word_to_id[word] for word in phrase.split()] for phrase in phrases],
    dtype=torch.long, device=device,
)
print("Readable rows:", phrases)
states = torch.randn(2, 4, 6, device=device)
for name, tensor in (("scalar", scalar), ("vector", vector), ("token_ids", token_ids), ("states", states)):
    print(f"{name:9s}: rank={tensor.ndim}, shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")


### One sentence, two ways to hand it over

`states[0]` opens the first example and hands you its contents: `[4, 6]`.
`states[0:1]` hands you a **one-example batch**: `[1, 4, 6]`.
The same 24 values are present, but one result still has an example axis.

> **Visual to add — Open the folder or keep a one-folder tray**
> Draw two folders labelled "the cat sat on" and "the dog sat on", each
> containing four position cards with six feature marks. Contrast `states[0]`
> as loose cards labelled `[4, 6]` with `states[0:1]` as one folder still in
> a tray labelled `[1, 4, 6]`. Suggested caption: "Same values; different
> instructions for the next layer."

**Predict:** Will the values match after deliberately removing the slice's
singleton axis for comparison? The next equality check tests that without
recommending a blind `squeeze()` on every batch.


In [ ]:
# -- Measure integer indexing versus slicing ----------------------------------
# (2 examples, 4 positions, 6 features) -> (4, 6): an integer removes batch.
one_example_without_batch = states[0]
# (2, 4, 6) -> (1, 4, 6): a slice keeps a one-example batch for later layers.
one_example_batch = states[0:1]
# Squeeze (1, 4, 6) -> (4, 6) only to compare values, not to feed a batch layer.
same_values = torch.equal(one_example_without_batch, one_example_batch.squeeze(0))
print(f"states[0]   -> {tuple(one_example_without_batch.shape)} = (time, feature)")
print(f"states[0:1] -> {tuple(one_example_batch.shape)} = (batch, time, feature)")
print(f"Values match after explicit squeeze: {same_values}")
assert same_values


## 2 - Regrouping isn't moving

At each position, divide six features into **two bundles of three**.
Reshape can do that while preserving their order. To read one bundle across
all positions, move the bundle axis before position:

```text
states       [2, 4, 6]       example, position, feature
split_heads  [2, 4, 2, 3]    example, position, bundle, width
heads_first  [2, 2, 4, 3]    example, bundle, position, width
```

All three contain 48 values. `reshape` changes their grouping; `transpose`
moves axes. A direct reshape of `heads_first` back to `[2, 4, 6]` would
not undo the axis move. First transpose back, then join the bundles.

The next cell checks exact round-trip equality. "Head" here is only a name
for a bundle; there is no attention mechanism deciding anything.
Once regrouping is clear, another question appears: how do you reuse an
offset at every position without copying it by hand?


In [ ]:
# -- Prove value preservation while changing the address system ---------------
# (B=2, T=4, D=6) -> (2, 4, H=2, width=3): group each position's six
# features into two heads without mixing examples or positions.
split_heads = states.reshape(2, 4, 2, 3)
# (B, T, H, width) -> (B, H, T, width): give each head its own position grid.
heads_first = split_heads.transpose(1, 2)
# (B, H, T, width) -> (B, T, H, width) -> (B, T, D): undo the axis move
# before joining features; reshaping heads_first directly would mix their order.
restored = heads_first.transpose(1, 2).reshape(2, 4, 6)
print(f"states      {tuple(states.shape)}")
print(f"split_heads {tuple(split_heads.shape)}")
print(f"heads_first {tuple(heads_first.shape)}")
print(f"Round-trip values preserved: {torch.equal(states, restored)}")
assert torch.equal(states, restored)


## 3 - An offset needs to say where it varies

`feature_bias` holds `[0, 1, 2, 3, 4, 5]`: one offset for each feature,
reused across both examples and all four positions. For example, feature
index 4 gets `+4` everywhere.

`time_bias` holds `[0, 1, 2, 3]`: one offset per position. Position index 2
should get `+2` across **every** feature in both examples. Those are different
instructions even though both inputs look like short lists.

![Feature offsets align with the final axis, but four position offsets need shape one by four by one to repeat across examples and features.](../images/02-broadcast-by-role.png)

PyTorch aligns dimensions from the right. `[6]` meets the feature axis
correctly. `[4]` meets that same axis and fails because four isn't six.
`[1, 4, 1]` states the intended reuse: vary by position, repeat across
examples and features.

**Predict:** Which addition fails, and which reshaping expresses the repair?
The caught error and printed result answer both. Broadcasting doesn't
understand words such as "position"; you supply that meaning.


In [ ]:
# -- Show one valid broadcast and one visible shape failure -------------------
feature_bias = torch.arange(6, device=device, dtype=states.dtype)
# (2, 4, 6) + (6,) -> (2, 4, 6): reuse one feature offset at every position.
shifted = states + feature_bias
time_bias = torch.arange(4, device=device, dtype=states.dtype)
try:
    states + time_bias
except RuntimeError as error:
    failure = str(error).splitlines()[0]
else:
    raise AssertionError("Expected incompatible trailing dimensions to fail")
# (4 positions,) -> (1, 4, 1): singleton batch/feature axes tell broadcasting
# to reuse each position's offset across examples and features; result stays (2, 4, 6).
fixed = states + time_bias.view(1, 4, 1)
print(f"Feature broadcast: {tuple(states.shape)} + {tuple(feature_bias.shape)} -> {tuple(shifted.shape)}")
print(f"Visible failure: {failure}")
print(f"Named repair: {tuple(states.shape)} + {tuple(time_bias.view(1, 4, 1).shape)} -> {tuple(fixed.shape)}")


Offsets reuse values. Gates scale values. Neither combines six features into
ten candidate scores. For that, we need a weighted sum of contributions.


## 4 - Turn features down, or mix them into answers?

A **gate** is a separate volume knob on each feature. The six gate values
below range from 0.5 to 1.0; multiplying by them still leaves six features
at each position.

A **projection** mixes all six features into one score for each of ten
candidates. As a small pencil example, two features `[2, -1]` and one
candidate's weights `[3, 4]` give `2×3 + (-1)×4 = 2`. The real random
projection below sums six contributions for each candidate.

```text
element-wise: [2, 4, 6] * [6]     -> [2, 4, 6]
projection:   [2, 4, 6] @ [6, 10] -> [2, 4, 10]
```

The projection consumes the feature axis by summing it; examples and positions
remain separate. Its random scores, or **logits**, are not probabilities
or useful word predictions. The next cell prints both shapes.


In [ ]:
# -- Compare scaling each coordinate with contracting the feature axis --------
gate = torch.linspace(0.5, 1.0, 6, device=device)
# (B=2, T=4, D=6) * (D=6,) -> (2, 4, 6): scale features, don't mix them.
gated = states * gate
W_vocab = torch.randn(6, 10, device=device)
# (B, T, D) @ (D, V=10) -> (B, T, V): sum feature contributions into
# one score per possible token, keeping examples and positions separate.
logits = states @ W_vocab
print(f"Element-wise: {tuple(states.shape)} * {tuple(gate.shape)} -> {tuple(gated.shape)}")
print(f"Matrix multiply: {tuple(states.shape)} @ {tuple(W_vocab.shape)} -> {tuple(logits.shape)}")


## 5 - Check backward on five before trusting forty-eight values

Autograd can differentiate a wrong calculation perfectly. First give it one
we can check ourselves:

```text
input 3 -> multiply by adjustable weight 2 -> add adjustable bias -1
        -> intermediate 5 -> square -> objective 25
```

Near 5, the square changes at 10 times a small change in its input.
A weight change reaches that point multiplied by the input 3, so its
sensitivity is 30. A bias change goes through directly, so its sensitivity
is 10. That's the chain rule before the symbolic shorthand.

The next cell compares those two answers with `.grad`. Input, intermediate,
objective, parameter, and gradient are five different roles. Backward writes
30 and 10 beside the parameters; it does not turn the weight into 30.


In [ ]:
# -- Audit one autograd result against the chain rule -------------------------
x_value = torch.tensor(3.0)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
intermediate = w * x_value + b
objective = intermediate**2
objective.backward()
manual_dw = 2 * intermediate.detach() * x_value
manual_db = 2 * intermediate.detach()
print(f"autograd dw={w.grad.item():.1f}; manual dw={manual_dw.item():.1f}")
print(f"autograd db={b.grad.item():.1f}; manual db={manual_db.item():.1f}")
assert torch.allclose(w.grad, manual_dw) and torch.allclose(b.grad, manual_db)


### The graph can retire while its feedback stays

`.grad` is an adding notepad. With `weight=2`, the calculation `weight*3`
returns 6, but its sensitivity to the weight is 3. Backward writes 3;
backward on a **new** `weight*3` calculation adds another 3. The weight
itself remains 2 because no optimizer takes a step.

> **Visual to add — Two fresh receipts, one accumulating notepad**
> Show two separate `2×3=6` calculation receipts arriving in sequence at
> the same weight knob marked 2. Each receipt adds "+3" to `.grad`,
> making the notepad read 3, then 6; clearing makes it 0. Keep the knob
> fixed throughout. Suggested caption: "A fresh graph doesn't mean a
> fresh gradient buffer."

`zero_()` clears that buffer. `detach()` instead leaves the gradient route;
it does **not** make an independent storage copy. `inference_mode()` avoids
recording new routes inside its block. Neither clears existing gradients.
Read the next printout as three separate checks: accumulation, clearing,
and whether later results need gradients.


In [ ]:
# -- Measure accumulation, clearing, detach, and inference mode ---------------
weight = torch.tensor(2.0, requires_grad=True)
(weight * 3).backward()
first_gradient = weight.grad.item()
(weight * 3).backward()
accumulated_gradient = weight.grad.item()
weight.grad.zero_()
detached = (weight * 3).detach()
with torch.inference_mode():
    inferred = weight * 3
print(f"After one backward: {first_gradient:.1f}; after two: {accumulated_gradient:.1f}")
print(f"After zero_: {weight.grad.item():.1f}")
print(f"detach requires_grad={detached.requires_grad}; inference requires_grad={inferred.requires_grad}")


The scalar check worked. Now keep the same distinctions when IDs fetch
stored rows: input addresses are not the table, fetched values are not
extra parameters, and a differentiable score isn't automatically a useful
learning objective.


## 6 - Follow an address through the whole calculation

![Two four-word examples keep their example and position axes as IDs fetch six-feature rows and a projection produces ten candidate scores per position.](../images/02-word-addresses-and-axes.png)

ID 1 fetches `embedding.weight[1]`; ID 4 fetches row 4. ID 0 fetches
row 0, not a row of zeros. These rows start random. The two phrases share
three IDs, so lookup reuses those same stored rows.

| Role | Shape | Count |
|---|---|---:|
| Input IDs | `[2, 4]` | 8 integer addresses |
| Stored embedding weights | `[10, 6]` | 60 parameters |
| Fetched activations | `[2, 4, 6]` | 48 values for these inputs |
| Stored `lm_head.weight` | `[10, 6]` | 60 parameters |
| Candidate-score activations | `[2, 4, 10]` | 80 scores |

`Linear(6,10)` stores its weights as output-by-input, so its `[10,6]`
storage is transposed relative to the earlier `[6,10]` matrix notation.
It still mixes six inputs into ten scores at each position.

> **Visual to add - Registry IDs select stored sensor profiles**
> Put the two actual ID strips `[0,1,2,3]` and `[0,4,2,3]` beside a catalogue
> of ten six-channel profiles. Shared IDs select the same stored profile;
> retrieved copies form two strips of four six-channel profiles. Label
> catalogue "60 stored parameters", strips "48 activations". Don't invent
> random feature values or label columns with word meanings.
> Suggested caption: "A lookup input is an address, not a ten-number row."

The cell squares all scores and averages them only to make a scalar for a
wiring test. It prints gradients shaped like both parameter tables.
There is no correct next-word target and no optimizer step. Useful text
training needs a meaningful objective; matching gradient shapes doesn't
supply one.


In [ ]:
# -- Build an embedding-to-logits graph and inspect gradient shapes -----------
vocabulary_size = 10
embedding_size = 6
embedding = torch.nn.Embedding(vocabulary_size, embedding_size).to(device)
lm_head = torch.nn.Linear(embedding_size, vocabulary_size, bias=False).to(device)
# IDs (B=2, T=4) -> (2, 4, D=6): replace each address with its learned row.
embedded = embedding(token_ids)
# (B, T, D=6) -> (B, T, V=10): score candidate tokens at each position;
# Linear contracts only the final feature axis, not batch or time.
lm_logits = lm_head(embedded)
# (2, 4, 10) -> scalar: average all squared scores only to audit gradient wiring.
toy_loss = lm_logits.square().mean()
toy_loss.backward()
print(f"IDs {tuple(token_ids.shape)} -> embeddings {tuple(embedded.shape)} -> logits {tuple(lm_logits.shape)}")
print(f"Embedding gradient shape: {tuple(embedding.weight.grad.shape)}")
print(f"LM-head gradient shape: {tuple(lm_head.weight.grad.shape)}")


### Your turn - repartition the same six features

Keep two examples and four positions. Change only `YOUR_HEADS`: three
bundles give two features each; two bundles give three. Four equal bundles
cannot hold six features, so the divisibility check should reject them.

**Predict:** Will the supplied setting end in `(3,2)` or `(2,3)`? The
output prints bundle count before bundle width. This is bookkeeping,
not newly trained attention.


In [ ]:
# -- CHANGE THIS: alter head count while preserving feature count -------------
YOUR_HEADS = 3
assert states.shape[-1] % YOUR_HEADS == 0
# (B=2, T=4, D=6) -> (B, T, YOUR_HEADS, 6 // YOUR_HEADS): regroup
# each position's features; the divisibility check prevents losing values.
your_heads = states.reshape(2, 4, YOUR_HEADS, states.shape[-1] // YOUR_HEADS)
print(f"With {YOUR_HEADS} heads: {tuple(states.shape)} -> {tuple(your_heads.shape)}")


## The short version

- A shape counts values by role. An integer index can remove a role
  without changing the values you selected.
- Regroup with reshape; move axes with transpose. Undo both to recover
  the original layout, not just the original value count.
- Broadcasting aligns from the right. `[1,4,1]` is a statement about
  position offsets, not a lucky shape that happens to run.
- Matrix multiplication mixes contributions; element-wise multiplication
  scales each coordinate separately.
- Backward records sensitivities. The scalar audit gives weight feedback
  30 and bias feedback 10; the accumulation audit adds 3, then 3 again.
- Lookup takes IDs, uses stored rows, and returns activations. The last
  graph checks gradient wiring, not word prediction.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Shapes/dtypes/devices; indexing; regrouping and moving; broadcasting failure/repair; projection; manual/autograd agreement; accumulation; detach; inference; embedding-to-score gradient shapes |
| Explained | Parameters versus activations; graph versus gradient lifetime; stored `Linear` orientation |
| Later | Useful word objectives and token preparation in Stage 03; attention in Stage 04; sparse tensors, custom backward functions, and distributed devices aren't needed for this route |

**Next:** [Models and Training Loops](../03-models-and-training-loops/03-models-and-training-loops.ipynb)
returns to engine-load and temperature rows. Once the axes are clear, the
remaining problem is who owns the adjustable numbers and how to save them.
